In [2]:
#https://pygithub.readthedocs.io/en/stable/introduction.html
from github import Github
# Authentication is defined via github.Auth
from github import Auth
import pandas as pd
import numpy as np
import json 
from datetime import datetime, date
from collections import Counter
import plotly.express as px
import time
import pickle
intrinsic_people = ["@aaronchongth","@akash-roboticist","@andreasBihlmaier","@arjo129","@audrow","@azeey","@damon-oss","@faximan","@jennuine","@koonpeng","@kscottz","@luca-della-vedova","@marcoag","@mbordignon-intrinsic","@methylDragon","@mjcarroll","@mjeronimo","@mxgrey","@nuclearsandwich-ai","@quarkytale","@scpeters","@sloretz","@tfoote","@udaya2899","@xiyuoh","@Yadunund"]
org_name = "gazebosim"

In [10]:
this_year = 2025
last_year = 2024

In [11]:
# Grab the access token
with open('./tokens.json',"r") as json_data:
    tokens = json.loads(json_data.read())
    json_data.close()

auth = Auth.Token(tokens["Github"])

# Public Web Github
gh = Github(auth=auth)

In [12]:
org = gh.get_organization(org_name)
repos = org.get_repos()

In [13]:
def extract_contributions(gh, repo_name, start_date, end_date):
# Get github repo level stats for two date ranges
    repo = gh.get_repo(repo_name)
    prs = repo.get_pulls(state='closed', sort='created')
    year = []
    results = {}
    count = 0
    for pr in prs:
        if start_date < pr.closed_at.date() < end_date:
            year.append(pr)
            count += 1
            
    results["repo"] = repo_name
    results["prs"] = year
    results["start_date"] = start_date    
    results["end_date"] = end_date    
    results["users"] = []
    results["handles"] = []
    results["add"] = 0
    results["del"] = 0 
    results["comments"] = 0
    results["files"] = 0
    
    for pr in year:
        results["users"].append(pr.user.name)
        results["handles"].append(pr.user.login)
        results["files"] += pr.changed_files 
        results["del"] += pr.deletions
        results["add"] += pr.additions
        results["comments"] += pr.review_comments

    results["total_prs"] = count
    results["total_users"] = len(set(results["users"]))

    return results

In [14]:
# Create a list of github repos for an org
full_repo_list = []
i = 0
has_repos = True
while has_repos:
    next_repos = repos.get_page(i)
    if len(next_repos) > 0:
        i += 1
        full_repo_list += next_repos
    else:
        has_repos = False
        
print(full_repo_list)
print(len(full_repo_list))

[Repository(full_name="gazebosim/ros_gz"), Repository(full_name="gazebosim/design"), Repository(full_name="gazebosim/gz-rviz"), Repository(full_name="gazebosim/sdf_tutorials"), Repository(full_name="gazebosim/sdformat"), Repository(full_name="gazebosim/docs"), Repository(full_name="gazebosim/ign-acropolis"), Repository(full_name="gazebosim/ign-blueprint"), Repository(full_name="gazebosim/gz-citadel"), Repository(full_name="gazebosim/gz-cmake"), Repository(full_name="gazebosim/gz-common"), Repository(full_name="gazebosim/gz-fuel-tools"), Repository(full_name="gazebosim/gz-sim"), Repository(full_name="gazebosim/ign-go1"), Repository(full_name="gazebosim/gz-gui"), Repository(full_name="gazebosim/gz-launch"), Repository(full_name="gazebosim/gz-math"), Repository(full_name="gazebosim/gz-msgs"), Repository(full_name="gazebosim/gz-physics"), Repository(full_name="gazebosim/gz-plugin"), Repository(full_name="gazebosim/gz-rendering"), Repository(full_name="gazebosim/gz-rndf"), Repository(full_n

In [16]:
this_year_start = date(2025, 3, 11)
this_year_end = date(2025, 9, 11)
last_year_start = date(2024, 9, 10)
last_year_end = date(2025, 3, 10)
full_org_results = {}
fname = 'github_stats_{0}_{1}-{2}.pkl'.format(org_name,this_year,last_year)
for repo in full_repo_list:
    print("Extracting data for {0} from {1} to {2}".format(repo.full_name,this_year_start,this_year_end))
    this_year_results = extract_contributions(gh, repo.full_name, this_year_start, this_year_end)
    print("Extracting data for {0} from {1} to {2}".format(repo.full_name,last_year_start,last_year_end))
    last_year_results = extract_contributions(gh, repo.full_name, last_year_start, last_year_end)
    full_org_results[repo.name] = {}
    full_org_results[repo.name][this_year] = this_year_results
    full_org_results[repo.name][last_year] = last_year_results
    with open(fname,"wb") as file:
        pickle.dump(full_org_results, file)
        print("Wrote: {0}".format(fname))
    print("-----------------------------")
    time.sleep(10)


Extracting data for gazebosim/ros_gz from 2025-03-11 to 2025-09-11
Extracting data for gazebosim/ros_gz from 2024-09-10 to 2025-03-10
Wrote: github_stats_gazebosim_2025-2024.pkl
-----------------------------
Extracting data for gazebosim/design from 2025-03-11 to 2025-09-11
Extracting data for gazebosim/design from 2024-09-10 to 2025-03-10
Wrote: github_stats_gazebosim_2025-2024.pkl
-----------------------------
Extracting data for gazebosim/gz-rviz from 2025-03-11 to 2025-09-11
Extracting data for gazebosim/gz-rviz from 2024-09-10 to 2025-03-10
Wrote: github_stats_gazebosim_2025-2024.pkl
-----------------------------
Extracting data for gazebosim/sdf_tutorials from 2025-03-11 to 2025-09-11
Extracting data for gazebosim/sdf_tutorials from 2024-09-10 to 2025-03-10
Wrote: github_stats_gazebosim_2025-2024.pkl
-----------------------------
Extracting data for gazebosim/sdformat from 2025-03-11 to 2025-09-11
Extracting data for gazebosim/sdformat from 2024-09-10 to 2025-03-10
Wrote: github_

Extracting data for gazebosim/gz-usd from 2024-09-10 to 2025-03-10
Wrote: github_stats_gazebosim_2025-2024.pkl
-----------------------------
Extracting data for gazebosim/gz_pkg_create from 2025-03-11 to 2025-09-11
Extracting data for gazebosim/gz_pkg_create from 2024-09-10 to 2025-03-10
Wrote: github_stats_gazebosim_2025-2024.pkl
-----------------------------
Extracting data for gazebosim/garden_demo from 2025-03-11 to 2025-09-11
Extracting data for gazebosim/garden_demo from 2024-09-10 to 2025-03-10
Wrote: github_stats_gazebosim_2025-2024.pkl
-----------------------------
Extracting data for gazebosim/garden-tutorial-party from 2025-03-11 to 2025-09-11
Extracting data for gazebosim/garden-tutorial-party from 2024-09-10 to 2025-03-10
Wrote: github_stats_gazebosim_2025-2024.pkl
-----------------------------
Extracting data for gazebosim/ros_gz_project_template from 2025-03-11 to 2025-09-11
Extracting data for gazebosim/ros_gz_project_template from 2024-09-10 to 2025-03-10
Wrote: github

In [17]:
out =  None
with open(fname, 'rb') as file:
        out = pickle.load(file)      
print(len(out.keys()))
print(out.keys())

54
dict_keys(['ros_gz', 'design', 'gz-rviz', 'sdf_tutorials', 'sdformat', 'docs', 'ign-acropolis', 'ign-blueprint', 'gz-citadel', 'gz-cmake', 'gz-common', 'gz-fuel-tools', 'gz-sim', 'ign-go1', 'gz-gui', 'gz-launch', 'gz-math', 'gz-msgs', 'gz-physics', 'gz-plugin', 'gz-rendering', 'gz-rndf', 'gz-sensors', 'gz-tools', 'gz-transport', 'gazebo-classic', 'testing', 'gz-bazel', 'ign-dome', 'gz-edifice', 'gz-utils', '.github', 'gz-fortress', 'fortress_demo', 'gz-garden', 'gz-omni', 'gz-omni-meta', 'gz-mujoco', 'gz-usd', 'gz_pkg_create', 'garden_demo', 'garden-tutorial-party', 'ros_gz_project_template', 'gz-chrono', 'gz-test', 'gz-harmonic', 'harmonic_demo', 'gazebo_test_cases', 'gz-ionic', 'ionic_demo', 'rules_gazebo', 'connect-samples', 'gz-jetty', 'jetty_demo'])


In [18]:
# Do full org aggregation
to_agg = ["users","add","del","files"]

full_results = {}
full_results[this_year] = {}
full_results[last_year] = {}

first = True
for key in full_org_results.keys():
    if first:
        full_results[this_year] = {k: full_org_results[key][this_year][k] for k in to_agg}
        full_results[last_year] = {k: full_org_results[key][last_year][k] for k in to_agg}
        full_results[this_year]["prs"] = len(full_org_results[key][this_year]["prs"])
        full_results[last_year]["prs"] = len(full_org_results[key][last_year]["prs"])
        first = False
    else:        
        for a in to_agg:
            full_results[this_year][a] += full_org_results[key][this_year][a]
            full_results[last_year][a] += full_org_results[key][last_year][a]
            full_results[this_year]["prs"] += len(full_org_results[key][this_year]["prs"])
            full_results[last_year]["prs"] += len(full_org_results[key][last_year]["prs"])

full_results[last_year]["contributors"] = set(full_results[last_year]["users"])
full_results[last_year]["users"] = len(set(full_results[last_year]["users"]))

full_results[this_year]["contributors"] = set(full_results[this_year]["users"])
full_results[this_year]["users"] = len(set(full_results[this_year]["users"]))


print("Results for {0} ==> {1}".format(last_year,this_year))
print("-------------------------")

temp_this = {}
temp_last = {}
temp_change = {}

for k in full_results[this_year].keys():
    if k == "contributors":
        continue
    change = -100*(full_results[last_year][k]-full_results[this_year][k])/full_results[last_year][k]
    temp_this[k] =  full_results[this_year][k]
    temp_last[k] =  full_results[last_year][k]
    temp_change[k] = change
    print("{0:6s}| {1} : {2:<6} | {3} : {4:<6} | {5:4.2f}%".format(k,last_year,full_results[last_year][k],this_year,full_results[this_year][k],change))
print(full_results[this_year]["contributors"])

summary_results = pd.DataFrame(data=[temp_this,temp_last,temp_change])
summary_results.to_csv("{0}-{1}-{2}-GithubContribsSummary.csv".format(org_name,last_year,this_year))

Results for 2024 ==> 2025
-------------------------
users | 2024 : 70     | 2025 : 82     | 17.14%
add   | 2024 : 277209 | 2025 : 248297 | -10.43%
del   | 2024 : 204399 | 2025 : 64057  | -68.66%
files | 2024 : 3802   | 2025 : 6370   | 67.54%
prs   | 2024 : 2556   | 2025 : 3159   | 23.59%
{'Nguyễn Tấn Anh Khoa', 'Kimberly N. McGuire', 'Shashank', 'Ian Chen', 'Gabriel Augusto', 'Saurabh Kamat', 'Dheeraj Deevi', None, 'Sai Aravind', 'Leo Lin', 'Ansh Chablani', 'Yaswanth', 'Xiaochuan Ye', 'Nate Koenig', 'Chris', 'Oscmoar07', 'Guilhem Saurel', 'Khaled Gabr', 'Fabrice Larribe', 'Dmytro Shkarupa', 'Alon Nusem', 'Gaurav Kumar', 'Antoine Van Malleghem', 'Abhiroop ', 'Steve Peters', 'Marcel Jacobse', 'Alejandro Hernández Cordero', 'Tarun Ramakrishnan', 'Gopikrishnan K', 'Alessandro Sofia', 'Maksim Derbasov', 'Avisheet Srivastava', 'Emmanuel M', 'Shyam Ganatra ', 'Michael Carroll', 'Afereti Pama', 'Jasper van Brakel', 'Øystein Sture', 'Michael Carlstrom', 'Mohamed Hassan', 'Shameek Ganguly', 'Dav

In [19]:
target = "add"
agg_result = []
for key in full_org_results.keys():
    a = full_org_results[key][this_year][target]
    b = full_org_results[key][last_year][target]
    delta = 0.00
    if b > 0:
        delta = (-100.0*(b-a)/b)
    temp = {}
    temp["name"] = key
    temp[this_year] = a
    temp[last_year] = b
    temp["change"] = delta
    agg_result.append(temp)
    
newlist = sorted(agg_result, key=lambda d: d[this_year])
newlist.reverse()
print("Results for '{0}' across {1} org".format(target,org))
print("-------------------------------------------------------------------")
for i in newlist:  
    print("{0:24s}| 2023: {1:<8} | 2024: {2:<8} | delta: {3:4.2f}%".format(i["name"][:24],
                                                                           i[last_year],
                                                                           i[this_year],
                                                                           i["change"]))
    
df = pd.DataFrame(data=newlist)
df.to_csv("{0}-{1}-{2}-NewLines.csv".format(org_name,last_year,this_year))

Results for 'add' across Organization(login="gazebosim") org
-------------------------------------------------------------------
gz-sim                  | 2023: 18938    | 2024: 130578   | delta: 589.50%
gz-transport            | 2023: 2438     | 2024: 47623    | delta: 1853.36%
gz-common               | 2023: 305      | 2024: 12226    | delta: 3908.52%
ros_gz                  | 2023: 11655    | 2024: 11708    | delta: 0.45%
gz-physics              | 2023: 1936     | 2024: 11195    | delta: 478.25%
jetty_demo              | 2023: 0        | 2024: 6491     | delta: 0.00%
sdformat                | 2023: 17823    | 2024: 4656     | delta: -73.88%
gz-gui                  | 2023: 743      | 2024: 4447     | delta: 498.52%
gz-rendering            | 2023: 6828     | 2024: 3925     | delta: -42.52%
gz-math                 | 2023: 3777     | 2024: 3509     | delta: -7.10%
gz-sensors              | 2023: 1265     | 2024: 2442     | delta: 93.04%
gz-cmake                | 2023: 955      | 2024: 2

In [21]:
target = "del"

agg_result = []
for key in full_org_results.keys():
    a = full_org_results[key][this_year][target]
    b = full_org_results[key][last_year][target]
    delta = 0.00
    if b > 0:
        delta = (-100.0*(b-a)/b)
    temp = {}
    temp["name"] = key
    temp[this_year] = a
    temp[last_year] = b
    temp["change"] = delta
    agg_result.append(temp)
    
newlist = sorted(agg_result, key=lambda d: d[this_year])
newlist.reverse()
print("PR count by year")
print("Results for '{0}' across ROS 2 org".format(target))
print("-------------------------------------------------------------------")
for i in newlist:  
    print("{0:24s}| 2023: {1:<8} | 2024: {2:<8} | delta: {3:4.2f}%".format(i["name"][:24],
                                                                           i[last_year],
                                                                           i[this_year],
                                                                           i["change"]))
df = pd.DataFrame(data=newlist)
df.to_csv("{0}-{1}-{2}-PRS.csv".format(org,last_year,this_year))

PR count by year
Results for 'del' across ROS 2 org
-------------------------------------------------------------------
gz-transport            | 2023: 1152     | 2024: 39190    | delta: 3301.91%
gz-sim                  | 2023: 3112     | 2024: 9883     | delta: 217.58%
gz-gui                  | 2023: 901      | 2024: 2668     | delta: 196.12%
gz-common               | 2023: 137      | 2024: 2538     | delta: 1752.55%
sdformat                | 2023: 3685     | 2024: 2493     | delta: -32.35%
gz-physics              | 2023: 784      | 2024: 986      | delta: 25.77%
gz-msgs                 | 2023: 223      | 2024: 872      | delta: 291.03%
gz-cmake                | 2023: 455      | 2024: 837      | delta: 83.96%
gz-math                 | 2023: 4276     | 2024: 807      | delta: -81.13%
ros_gz                  | 2023: 2284     | 2024: 802      | delta: -64.89%
gz-rendering            | 2023: 718      | 2024: 737      | delta: 2.65%
gz-sensors              | 2023: 274      | 2024: 392     

In [24]:
target = "prs"

agg_result = []
for key in full_org_results.keys():
    a = len(full_org_results[key][this_year][target])
    b = len(full_org_results[key][last_year][target])
    delta = 0.00
    if b > 0:
        delta = (-100.0*(b-a)/b)
    temp = {}
    temp["name"] = key
    temp[this_year] = a
    temp[last_year] = b
    temp["change"] = delta
    agg_result.append(temp)
    
newlist = sorted(agg_result, key=lambda d: d[this_year])
newlist.reverse()
print("PR count by year")
print("Results for '{0}' across Gzebo org".format(target))
print("-------------------------------------------------------------------")
for i in newlist:  
    print("{0:24s}| Prev Six Months: {1:<8} | Last Six Months: {2:<8} | delta: {3:4.2f}%".format(i["name"][:24],
                                                                           i[last_year],
                                                                           i[this_year],
                                                                           i["change"]))
df = pd.DataFrame(data=newlist)
df.to_csv("{0}-{1}-{2}-PRS-six-month.csv".format(org,last_year,this_year))

PR count by year
Results for 'prs' across Gzebo org
-------------------------------------------------------------------
gz-sim                  | Prev Six Months: 105      | Last Six Months: 157      | delta: 49.52%
gz-transport            | Prev Six Months: 39       | Last Six Months: 123      | delta: 215.38%
ros_gz                  | Prev Six Months: 72       | Last Six Months: 63       | delta: -12.50%
gz-physics              | Prev Six Months: 34       | Last Six Months: 54       | delta: 58.82%
gz-gui                  | Prev Six Months: 15       | Last Six Months: 51       | delta: 240.00%
gz-rendering            | Prev Six Months: 46       | Last Six Months: 44       | delta: -4.35%
gz-msgs                 | Prev Six Months: 28       | Last Six Months: 37       | delta: 32.14%
gz-common               | Prev Six Months: 24       | Last Six Months: 35       | delta: 45.83%
gz-sensors              | Prev Six Months: 24       | Last Six Months: 34       | delta: 41.67%
docs         